Earning 50K+? Income with a Fairness Check

Decision Tree Classification Project
Student details:
Name      :Krish Vekariya
Roll No.  :B-61
Div       :D
Enrollment:  24004501210477

import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    RocCurveDisplay
)

warnings.filterwarnings("ignore")

print("All libraries loaded!")

LOAD DATASET

In [ ]:
df = pd.read_csv("adult.csv")

print("Dataset shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())

UNDERSTAND THE DATA

In [ ]:
print("Dataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe(include="all").T)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nColumn names:")
print(df.columns.tolist())

CHECK TARGET COLUMN

In [ ]:
TARGET = "income"

df[TARGET] = df[TARGET].astype("string").str.strip()

print("Income counts:")
print(df[TARGET].value_counts(dropna=False))

print("\nIncome percentages:")
print((df[TARGET].value_counts(normalize=True) * 100).round(2))

CLEAN THE DATASET

In [ ]:
df = df.replace(r"^\s*\?\s*$", np.nan, regex=True)

df = df.drop_duplicates()

df = df.dropna(subset=[TARGET])

# Remove spaces around text values
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].str.strip()

# Fill missing numeric and categorical values
for col in df.columns:
    if col == TARGET:
        continue

    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        mode_values = df[col].mode(dropna=True)

        if len(mode_values) > 0:
            df[col] = df[col].fillna(mode_values.iloc[0])

print("Missing values remaining:", df.isnull().sum().sum())
print("Cleaned dataset shape:", df.shape)

display(df.head())

**CHARTS**

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x=TARGET)
plt.title("Income Class Distribution")
plt.xlabel("Income")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()

df[num_cols].hist(figsize=(14, 10), bins=30)
plt.suptitle("Distribution of Numeric Features")
plt.tight_layout()
plt.show()

In [ ]:
if "age" in df.columns:
    plt.figure(figsize=(7, 4))
    sns.boxplot(data=df, x=TARGET, y="age")
    plt.title("Age Distribution by Income")
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(
    df[num_cols].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

CREATE NEW FEATURES 

In [ ]:
# Step 7: Create a New Feature, Encode Data and Select Features

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
import pandas as pd

# 1) Create a new feature from existing columns
df["hours_over_40"] = (df["hours.per.week"] - 40).clip(lower=0)

print("New feature created: hours_over_40")
display(df[["hours.per.week", "hours_over_40"]].head())

# 2) Separate X (inputs) and y (target)
TARGET = "income"

X = df.drop(columns=[TARGET])

le_y = LabelEncoder()
y = le_y.fit_transform(df[TARGET].astype(str).str.strip())

print("Classes:", list(le_y.classes_))
# 0 = <=50K, 1 = >50K (if these are the two sorted class labels)

# 3) One-hot encoding: convert text columns into 0/1 columns
X = pd.get_dummies(X, drop_first=True, dtype=int)

print("Encoded dataset shape:", X.shape)
display(X.head())

# 4) Feature selection using SelectKBest
k = min(15, X.shape[1])

selector = SelectKBest(score_func=f_classif, k=k)
selector.fit(X, y)

scores = pd.Series(
    selector.scores_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop 20 feature scores:")
display(scores.head(20))

# Keep the selected features
selected_features = scores.head(k).index.tolist()
X = X[selected_features]

feature_names = list(X.columns)

print("\nSelected features:")
print(feature_names)

print("\nFinal feature shape:", X.shape)

Split data, select features and scale

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train_raw.shape)
print("Testing shape:", X_test_raw.shape)

# Select up to 15 features using training data only
k = min(15, X_train_raw.shape[1])

selector = SelectKBest(score_func=f_classif, k=k)

X_train_selected = selector.fit_transform(X_train_raw, y_train)
X_test_selected = selector.transform(X_test_raw)

feature_names = X_train_raw.columns[selector.get_support()].tolist()

print("\nSelected features:")
print(feature_names)

# Scale using training data only
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train_selected)
X_test = scaler.transform(X_test_selected)

print("\nFinal training shape:", X_train.shape)
print("Final testing shape:", X_test.shape)

**Train the baseline Logistic Regression model**

In [ ]:
results = []

def evaluate(name, model, show=False):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_test, y_pred, labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    row = {
        "Setting": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0),
        "Specificity": specificity,
        "AUC": roc_auc_score(y_test, probabilities)
    }

    if show:
        ConfusionMatrixDisplay(
            confusion_matrix(y_test, y_pred),
            display_labels=["<=50K", ">50K"]
        ).plot(cmap="Blues")

        plt.title(name)
        plt.tight_layout()
        plt.show()

        print("\nClassification Report:")
        print(classification_report(
            y_test,
            y_pred,
            labels=[0, 1],
            target_names=["<=50K", ">50K"],
            zero_division=0
        ))

        RocCurveDisplay.from_predictions(y_test, probabilities)
        plt.title("ROC Curve - " + name)
        plt.tight_layout()
        plt.show()

    print(row)
    return row

baseline_model = LogisticRegression(max_iter=1000)

results.append(evaluate("Baseline", baseline_model, show=True))

TUNE LOGISTIC REGRESSION

In [ ]:


from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score,
    precision_score, recall_score, f1_score,
    roc_auc_score, RocCurveDisplay
)
import pandas as pd
import matplotlib.pyplot as plt

# A) SCALE FEATURES (fit on training data only)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train_raw)
X_test = scaler.transform(X_test_raw)

# Store results from each setting
results = []

# Evaluation function
def evaluate(name, model, show=False):
    model.fit(X_train, y_train)

    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(
        y_test, pred, labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    row = {
        "Setting": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(
            y_test, pred, zero_division=0
        ),
        "Recall": recall_score(
            y_test, pred, zero_division=0
        ),
        "F1": f1_score(y_test, pred, zero_division=0),
        "Specificity": specificity,
        "AUC": roc_auc_score(y_test, proba)
    }

    if show:
        ConfusionMatrixDisplay(
            confusion_matrix(y_test, pred),
            display_labels=["<=50K", ">50K"]
        ).plot(cmap="Blues")

        plt.title(name)
        plt.tight_layout()
        plt.show()

        print("\nClassification Report:")
        print(classification_report(
            y_test, pred,
            target_names=["<=50K", ">50K"],
            zero_division=0
        ))

        RocCurveDisplay.from_predictions(y_test, proba)
        plt.title("ROC Curve - " + name)
        plt.tight_layout()
        plt.show()

    print(row)
    return row


# B) COMPARE class_weight=None VS class_weight="balanced"
for cw in [None, "balanced"]:
    model = LogisticRegression(
        max_iter=1000,
        class_weight=cw
    )

    results.append(
        evaluate(f"class_weight={cw}", model)
    )


# C) PRINT INTERCEPT AND COEFFICIENTS
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

print("\nIntercept:", lr.intercept_)

coef = pd.Series(
    lr.coef_[0],
    index=feature_names
).sort_values()

print("\nThree strongest negative features:")
print(coef.head(3))

print("\nThree strongest positive features:")
print(coef.tail(3))

coef.plot(kind="barh", figsize=(8, 6))
plt.title("Logistic Regression Coefficients")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.show()


# D) COMPARE DECISION THRESHOLDS
proba = lr.predict_proba(X_test)[:, 1]

print("\nDecision Threshold Comparison:")

for t in [0.3, 0.5, 0.7]:
    pred = (proba >= t).astype(int)

    precision = precision_score(
        y_test, pred, zero_division=0
    )
    recall = recall_score(
        y_test, pred, zero_division=0
    )

    print(
        f"Threshold: {t} | "
        f"Precision: {precision:.3f} | "
        f"Recall: {recall:.3f}"
    )


# E) DISPLAY COMPARISON OF MODEL SETTINGS
results_df = pd.DataFrame(results).round(3)

print("\nModel Comparison:")
display(results_df)

COMPARE SETTINGS AND SELECT FINAL MODEL

In [ ]:


import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# 1. Create a comparison table
res = pd.DataFrame(results).round(3)

print("Model Comparison Table:")
display(res)

# 2. Plot model performance
res.set_index("Setting")[
    ["Accuracy", "Precision", "Recall", "F1", "AUC"]
].plot(kind="bar", figsize=(12, 5))

plt.title("Comparison of Logistic Regression Settings")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 3. Select the best setting using F1-score
# Use AUC as a tie-breaker
best = res.sort_values(
    ["F1", "AUC"],
    ascending=False
).iloc[0]

best_setting = best["Setting"]

print("Best setting:", best_setting)
print("Best F1-score:", best["F1"])
print("Best AUC:", best["AUC"])

# 4. Create the final model using the selected setting
if best_setting == "class_weight=balanced":
    final_model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    )
else:
    final_model = LogisticRegression(
        max_iter=1000,
        class_weight=None
    )

# 5. Evaluate the final model
final_results = evaluate(
    "FINAL MODEL",
    final_model,
    show=True
)

print("\nFinal Model Results:")
display(pd.DataFrame([final_results]).round(3))

# 6. Save the comparison table
res.to_csv("p38_model_comparison.csv", index=False)

print("Model comparison saved to p38_model_comparison.csv")

 Use three sample rows from the original encoded features

In [ ]:
# Use three sample rows from the original encoded 
new = X.iloc[[0, 1, 2]].copy()

display(new)

# Apply the fitted feature selector and scaler
new_selected = selector.transform(new)
new_scaled = scaler.transform(new_selected)

pred = final_model.predict(new_scaled)
proba = final_model.predict_proba(new_scaled)[:, 1]

prediction_results = pd.DataFrame({
    "Sample": ["Sample 1", "Sample 2", "Sample 3"],
    "Predicted Income": [
        ">50K" if value == 1 else "<=50K"
        for value in pred
    ],
    "Probability of >50K": proba.round(3)
})

display(prediction_results)